# Backend Benchmark Exploration

Use this notebook first. It runs the ChromaDB vs pgvector benchmark and shows pandas DataFrames for speed, retrieval quality, scores, and ranking parity. Nothing is written to `evaluation_results.json` unless you run the optional save cell at the end.

## Before Running

- Make sure `.env` points `PDF_SOURCE_DIR` at your PDFs.
- Make sure chunks already exist under `data/interim/chunks/`; if not, run `python pipeline.py extract` first.
- For pgvector, run `docker compose up -d`, `alembic upgrade head`, and `python scripts/check_pgvector.py`.
- If imports fail, check the kernel path in the next cell. VS Code/Jupyter must use the same environment where the project dependencies are installed.

In [ ]:
import sys

print(sys.executable)
print(sys.version)
# If this path is not your project environment, switch kernels in VS Code.
# To install dependencies into this exact kernel, run:
# %pip install -r requirements.txt

In [ ]:
import pandas as pd

from benchmark_metrics import (
    run_backend_benchmark,
    summarise_results,
    write_results_json,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.precision", 6)

## Parameters

For a quick smoke test, set `MAX_CHUNKS` or `MAX_QUERIES` to a small number. For the professor's spec, eventually use 20 representative queries and `QUERY_REPEATS = 3`.

In [ ]:
BACKENDS = ("chroma", "pgvector")
BATCH_SIZE = 256
QUERY_REPEATS = 3
K = 5
MAX_CHUNKS = None
MAX_QUERIES = None

{
    "backends": BACKENDS,
    "batch_size": BATCH_SIZE,
    "query_repeats": QUERY_REPEATS,
    "k": K,
    "max_chunks": MAX_CHUNKS,
    "max_queries": MAX_QUERIES,
}

## Run Benchmark

This returns raw pandas DataFrames. It uses temporary benchmark collections and cleans them up after each run.

In [ ]:
results = run_backend_benchmark(
    backends=BACKENDS,
    batch_size=BATCH_SIZE,
    query_repeats=QUERY_REPEATS,
    k=K,
    max_chunks=MAX_CHUNKS,
    max_queries=MAX_QUERIES,
    cleanup=True,
)

summaries = summarise_results(results)
results["parameters"]

In [ ]:
pd.DataFrame(results["errors"])

## Ingestion Speed

`total_embed_and_store_seconds` matches the professor's requested embed + store timing. `store_add_seconds` isolates just the backend write cost.

In [ ]:
display(summaries.get("ingestion_summary", pd.DataFrame()))
display(results["ingestion_batches"])

## Query Latency

The professor asked for three runs and median latency. The summary table includes median, mean, p95, min, and max.

In [ ]:
display(summaries.get("query_latency_summary", pd.DataFrame()))
display(results["query_latency"].head(30))

## Recall@5, Precision@5, and MRR

In [ ]:
display(summaries.get("retrieval_quality_summary", pd.DataFrame()))
display(results["retrieval_quality"])

## Scores and Ranking Parity

This checks whether both backends return the same top-k chunk IDs in the same order.

In [ ]:
display(summaries.get("score_summary", pd.DataFrame()))
display(summaries.get("ranking_parity_summary", pd.DataFrame()))
display(results["ranking_parity"])
display(results["topk_results"].head(30))

## Metadata Filters

The professor asks for company, year, and sector filtering. This table shows which `where` filters are available from the current chunk metadata.

In [ ]:
display(results["metadata_filters"])
display(results["query_latency"].groupby(["backend", "filter_name"], as_index=False)["seconds"].median())

## Deployment Complexity

Concrete counts for setup complexity: extra services, Docker Compose line count, and clean-machine backend setup steps.

In [ ]:
display(summaries.get("deployment_complexity_summary", pd.DataFrame()))

## Code Legibility

Concrete counts for each backend implementation. If `radon` is installed in the kernel, this also includes cognitive-complexity scores.

In [ ]:
display(summaries.get("code_legibility_summary", pd.DataFrame()))

## Optional Save

Only run this after the tables look right.

In [ ]:
# write_results_json(results, "evaluation_results.json")